# Colab Setup

**For Colab users:** Uncomment and run the cell below to mount Drive.  
**For local users:** Skip this cell.

In [1]:
# Uncomment for Colab:
#from google.colab import drive
#drive.mount('/content/drive')

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [ ]:
# --- 1. Verify and install dependencies ---
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  ✅ {package} is installed.")
    except ImportError:
        print(f"  ❌ {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  ✅ {package} is now installed.")
        except Exception as e:
            print(f"  ❌ Failed to install {package}: {e}")

# Special check for PyTorch CUDA
print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  ❌ PyTorch is not installed.")

print("Verification complete.")

Verifying and installing missing packages if necessary...
  ✅ numpy is installed.
  ✅ matplotlib is installed.
  ✅ librosa is installed.
  ✅ tqdm is installed.
  ✅ sklearn is installed.
  ✅ stempeg is installed.
  ✅ torch is installed.
  ✅ torchvision is installed.
  ✅ torchaudio is installed.

--- PyTorch CUDA status ---
  ⚠️ PyTorch is installed, but CUDA is NOT available.
Verification complete.


## Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [3]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import importlib

# Detect environment and find Project Root
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # In Colab: search for project folder in Drive
    possible_roots = [
        Path('/content/drive/MyDrive/Final_Project_Deep_Learning'),
        Path('/content/drive/MyDrive/Final_Project_Deep_Lea'),
        Path('/content/drive/MyDrive/Colab Notebooks/Final_Project_Deep_Learning')
    ]
    PROJECT_ROOT = None
    for path in possible_roots:
        if path.exists() and (path / 'local_main.ipynb').exists():
            PROJECT_ROOT = path
            break
    
    if PROJECT_ROOT is None:
        print("❌ Error: Could not find 'Final_Project_Deep_Learning' in Drive.")
        print("   Please verify the folder name and that you mounted Drive.")
        PROJECT_ROOT = Path.cwd()
    else:
        print(f"✅ Found Project Root: {PROJECT_ROOT}")
else:
    # Local: use current working directory
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'local_main.ipynb').exists():
        for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
            if (p / 'local_main.ipynb').exists():
                PROJECT_ROOT = p
                break

os.chdir(PROJECT_ROOT)

# Define data and checkpoint directories
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Add to sys.path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import project modules
import models.utils as utils
importlib.reload(utils)
from models import model_A as ma

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nConfiguration Complete:")
print(f"   - Device: {device}")
print(f"   - Working Directory: {os.getcwd()}")
print(f"   - Data Directory: {DATA_DIR}")
print(f"   - Checkpoints Directory: {CHECKPOINT_DIR}")


Configuration Complete:
   - Device: cpu
   - Working Directory: c:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning
   - Data Directory: c:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning\data
   - Checkpoints Directory: c:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning\checkpoints


## MUSDB18 Setup

**Quick Start:**
1. Download MUSDB18
2. Extract to project folder as `musdb18/`
3. Run preprocessing below

**Expected structure:**
```
musdb18/
  train/     (~100 songs)
  valid/     (~14 songs)
  test/      (~50 songs)
```

In [4]:
# ============================================================================
# MUSDB18 PATH CONFIGURATION
# ============================================================================
# Auto-detect musdb18 folder in project directory
MUSDB18_PATH = PROJECT_ROOT / "musdb18"

if not MUSDB18_PATH.exists():
    print("⚠️  MUSDB18 folder not found!")
    print(f"   Expected location: {MUSDB18_PATH}")
    print("")
    print("📥 Download MUSDB18:")
    print("   1. Register at https://zenodo.org/record/1438122")
    print("   2. Download MUSDB18-HQ.zip (~22GB)")
    print("   3. Extract to project folder as 'musdb18'")
    print("")
    MUSDB18_PATH = None
else:
    # Check for required subfolders
    train_dir = MUSDB18_PATH / "train"
    valid_dir = MUSDB18_PATH / "valid"
    test_dir = MUSDB18_PATH / "test"
    
    if train_dir.exists() and test_dir.exists():
        num_train = len(list(train_dir.iterdir()))
        num_valid = len(list(valid_dir.iterdir())) if valid_dir.exists() else 0
        num_test = len(list(test_dir.iterdir()))
        print(f"✅ MUSDB18 dataset found: {MUSDB18_PATH}")
        print(f"   Train: {num_train} tracks")
        print(f"   Valid: {num_valid} tracks")
        print(f"   Test: {num_test} tracks")
    else:
        print(f"⚠️  Found musdb18 folder but missing train/test subfolders")
        print(f"   Path: {MUSDB18_PATH}")
        MUSDB18_PATH = None

# Initialize file lists (will be populated after preprocessing)
mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

✅ MUSDB18 dataset found: c:\Users\amita\source\repos\Deep learning on computational accelerators\Final_Project_Deep_Learning\musdb18
   Train: 100 tracks
   Valid: 16 tracks
   Test: 50 tracks


## Data Preprocessing

Create 4 chunk size versions (6s, 8s, 10s, 12s) to test different architectures. Curriculum learning: Stage 1 (easy), Stage 2 (realistic).

In [5]:
# ============================================================================
# DATA PREPROCESSING: MUSDB18 → CHUNKED TRAINING DATA (MULTIPLE CHUNK SIZES)
# ============================================================================

PROCESS_DATA = True  # Set to False to skip preprocessing (use existing data)
OVERLAP = 0.5  # Overlap ratio (50% overlap for more training data)
SAMPLE_RATE = 22050  # Target sample rate for all audio

# Chunk configurations for preset comparison
# Each preset architecture requires specific chunk duration
CHUNK_CONFIGS = [
    {'duration': 6.0, 'name': 'chunks_6s'},   # For: Small (4L/128F), Deepest-6 (6L/256F)
    {'duration': 8.0, 'name': 'chunks_8s'},   # For: Medium (4L/256F), Deeper-5 (5L/256F)
    {'duration': 10.0, 'name': 'chunks_10s'}, # For: Large (4L/512F), Very Deep-7 (7L/256F)
    {'duration': 12.0, 'name': 'chunks_12s'}, # For: Ultra Deep-8 (8L/256F)
]

if PROCESS_DATA and MUSDB18_PATH:
    print(f"\n{'='*70}")
    print(f"PREPROCESSING {len(CHUNK_CONFIGS)} CHUNK CONFIGURATIONS")
    print(f"{'='*70}\n")
    
    for idx, config in enumerate(CHUNK_CONFIGS, 1):
        chunk_dir = DATA_DIR / config['name']
        
        # Skip if already exists
        if (chunk_dir / "stage1" / "train" / "mixture").exists():
            print(f"[{idx}/{len(CHUNK_CONFIGS)}] ⏭️  {config['name']} already exists, skipping...")
            continue
        
        print(f"\n[{idx}/{len(CHUNK_CONFIGS)}] 🔄 Processing {config['duration']}s chunks...")
        print(f"Output: {chunk_dir}")
        
        # Run preprocessing
        preprocessing_stats = utils.preprocess_musdb18(
            musdb18_path=MUSDB18_PATH,
            output_dir=chunk_dir,
            chunk_duration=config['duration'],
            overlap=OVERLAP,
            sample_rate=SAMPLE_RATE,
            stage1_ratio=0.7,
            train_ratio=0.7,
            val_ratio=0.15,
            test_ratio=0.15
        )
        
        print(f"✅ {config['name']} complete!")
    
    print(f"\n{'='*70}")
    print("🎯 ALL PREPROCESSING COMPLETE!")
    print(f"{'='*70}")
    print("\nData structure:")
    print("  data/")
    for config in CHUNK_CONFIGS:
        print(f"    {config['name']}/")
        print(f"      stage1/ (train/val/test → mixture/target)")
        print(f"      stage2/ (train/val/test → mixture/target)")
    
    # Verify one chunk size as example
    print(f"\n{'='*70}")
    print("VERIFICATION (chunks_8s example)")
    print(f"{'='*70}\n")
    
    sample_dir = DATA_DIR / "chunks_8s"
    if sample_dir.exists():
        for stage in ['stage1', 'stage2']:
            print(f"{stage.upper()}:")
            for split in ['train', 'val', 'test']:
                mix_dir = sample_dir / stage / split / "mixture"
                if mix_dir.exists():
                    n = len(list(mix_dir.glob("*.npy")))
                    print(f"  {split:5s}: {n:,} chunks")
            print()

elif PROCESS_DATA and not MUSDB18_PATH:
    print("⚠️  Cannot process data: MUSDB18_PATH not set")
    print("   Please extract the dataset to the 'musdb18' folder")
    
else:
    print("ℹ️  Data preprocessing skipped (PROCESS_DATA = False)")
    print("   Using existing preprocessed data...")



PREPROCESSING 4 CHUNK CONFIGURATIONS

[1/4] ⏭️  chunks_6s already exists, skipping...
[2/4] ⏭️  chunks_8s already exists, skipping...
[3/4] ⏭️  chunks_10s already exists, skipping...
[4/4] ⏭️  chunks_12s already exists, skipping...

🎯 ALL PREPROCESSING COMPLETE!

Data structure:
  data/
    chunks_6s/
      stage1/ (train/val/test → mixture/target)
      stage2/ (train/val/test → mixture/target)
    chunks_8s/
      stage1/ (train/val/test → mixture/target)
      stage2/ (train/val/test → mixture/target)
    chunks_10s/
      stage1/ (train/val/test → mixture/target)
      stage2/ (train/val/test → mixture/target)
    chunks_12s/
      stage1/ (train/val/test → mixture/target)
      stage2/ (train/val/test → mixture/target)

VERIFICATION (chunks_8s example)

STAGE1:
  train: 3,168 chunks
  val  : 678 chunks
  test : 680 chunks

STAGE2:
  train: 1,429 chunks
  val  : 306 chunks
  test : 307 chunks



## Architecture Competition

Compare Small vs Medium architectures to find the best one. Each preset loads its optimal chunk duration.

In [ ]:
import importlib
import gc
importlib.reload(utils)

# ============================================================================
# AUTOMATED PRESET COMPARISON - Small vs Medium Only
# ============================================================================

PRESETS = [
    {
        'name': 'Small (6.0s, 4 layers, 128 filters)',
        'chunk_duration': 6.0,
        'num_layers': 4,
        'base_filters': 128,
    },
    {
        'name': 'Medium (8.0s, 4 layers, 256 filters)',
        'chunk_duration': 8.0,
        'num_layers': 4,
        'base_filters': 256,
    },
]

print(f"\n{'='*70}")
print(f"AUTOMATED PRESET COMPARISON - Testing {len(PRESETS)} architectures")
print(f"(Small vs Medium)")
print(f"{'='*70}\n")

results = []

# Test both presets
for idx, preset in enumerate(PRESETS, start=1):
    print(f"\n[{idx}/{len(PRESETS)}] Testing: {preset['name']}")
    print(f"-" * 70)

    # Configure
    OVERFIT_CONFIG = utils.get_overfit_config(
        chunk_duration=preset['chunk_duration'],
        num_layers=preset['num_layers']
    )
    BASE_FILTERS = preset['base_filters']
    MODEL_NAME = preset['name']

    print(f"Config: Chunk={OVERFIT_CONFIG['chunk_duration']}s | Layers={OVERFIT_CONFIG['num_layers']} | Filters={BASE_FILTERS}")

    # Build model
    overfit_device = device
    overfit_processor = utils.AudioProcessor(device=overfit_device)

    overfit_model = ma.TimeFrequencyDomainUNet(
        in_channels=1,
        out_channels=1,
        base_filters=BASE_FILTERS,
        num_layers=OVERFIT_CONFIG['num_layers'],
        batchnorm=True,
        dropout=0.0
    ).to(overfit_device)

    total_params = sum(p.numel() for p in overfit_model.parameters())
    print(f"Parameters: {total_params:,}")

    # Train
    overfit_loss_fn = nn.MSELoss()
    overfit_optimizer = optim.Adam(overfit_model.parameters(), lr=OVERFIT_CONFIG['learning_rate'])

    overfit_ckpt = CHECKPOINT_DIR / f"debug_overfit_{idx}_preset.pth"

    # Clean old checkpoint
    if overfit_ckpt.exists():
        overfit_ckpt.unlink()
    epochs_dir = CHECKPOINT_DIR / f"debug_overfit_{idx}_preset_epochs"
    if epochs_dir.exists():
        import shutil
        shutil.rmtree(epochs_dir)

    # Point to correct chunk directory for this preset
    chunk_duration_str = f"{preset['chunk_duration']:.0f}s"
    preset_data_dir = DATA_DIR / f"chunks_{chunk_duration_str}"

    # Run training with error handling
    try:
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(preset_data_dir),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )

        # Extract metrics
        if history_overfit and 'train_loss' in history_overfit:
            final_train_loss = history_overfit['train_loss'][-1]
            final_val_loss = history_overfit['val_loss'][-1]
            min_val_loss = min(history_overfit['val_loss'])
            num_epochs = len(history_overfit['train_loss'])
        else:
            final_train_loss = float('inf')
            final_val_loss = float('inf')
            min_val_loss = float('inf')
            num_epochs = 0

        results.append({
            'name': MODEL_NAME,
            'final_train_loss': final_train_loss,
            'final_val_loss': final_val_loss,
            'min_val_loss': min_val_loss,
            'num_epochs': num_epochs,
            'params': total_params,
            'history': history_overfit,
            'ckpt_path': str(overfit_ckpt)
        })

        print(f"✅ Final Train Loss: {final_train_loss:.6f} | Final Val Loss: {final_val_loss:.6f} | Best Val: {min_val_loss:.6f}")

    except ZeroDivisionError as e:
        print(f"❌ ERROR: No valid chunks found for {preset['chunk_duration']}s duration")
        print(f"   The randomly selected song is shorter than {preset['chunk_duration']}s")
        print(f"   → SKIPPING this preset")
    except Exception as e:
        print(f"❌ ERROR during training: {type(e).__name__}: {str(e)[:100]}")
        print(f"   → SKIPPING this preset")
    finally:
        # Memory cleanup
        print(f"🧹 Cleaning up memory...")
        del overfit_model, overfit_optimizer, overfit_processor, overfit_loss_fn
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"✅ Memory cleaned!\n")

# ============================================================================
# COMPARISON AND WINNER ANNOUNCEMENT
# ============================================================================
print(f"\n\n{'='*70}")
print("RESULTS SUMMARY - Small vs Medium Comparison")
print(f"{'='*70}\n")

if results:
    import pandas as pd
    df_results = pd.DataFrame([
        {
            'Architecture': r['name'],
            'Final Train Loss': f"{r['final_train_loss']:.6f}",
            'Final Val Loss': f"{r['final_val_loss']:.6f}",
            'Best Val Loss': f"{r['min_val_loss']:.6f}",
            'Epochs': r['num_epochs'],
            'Parameters': f"{r['params']:,}"
        }
        for r in results
    ])

    print(df_results.to_string(index=False))

    # Find winner by lowest final val loss
    best_result = min(results, key=lambda x: x['final_val_loss'])
    print(f"\n🏆 BEST PRESET: {best_result['name']}")
    print(f"   Final Val Loss: {best_result['final_val_loss']:.6f}")
    print(f"   Parameters: {best_result['params']:,}")

    print(f"\n✅ Comparison complete! Best config ready for full training.")
else:
    print("⚠️  No presets completed successfully.")
    print("   Both presets either failed or were skipped.")


AUTOMATED PRESET COMPARISON - Testing 7 architectures
(Small → Medium → Large → Deeper-5 → Deepest-6 → Very Deep-7 → Ultra Deep-8)


[1/7] Testing: Small (6.0s, 4 layers, 128 filters)
----------------------------------------------------------------------
Config: Chunk=6.0s | Layers=4 | Filters=128
Parameters: 86,376,065
   Song #348: 1 chunks (6.0s each)
   Training from scratch on 1 random song (#348)...
[DEBUG] Using tqdm.notebook for progress bars.


Total Progress:   0%|          | 0/100 [00:00<?, ?it/s]

[DEBUG] Using tqdm.notebook for progress bars.


Ep 1 Training:   0%|          | 0/1 [00:00<?, ?it/s]

🧹 Cleaning up memory...
✅ Memory cleaned!



KeyboardInterrupt: 

## Winner Selection

Extract best architecture and delete unused preset checkpoints.

In [ ]:
# ============================================================================
# CLEANUP: DELETE UNUSED PRESET CHECKPOINTS
# ============================================================================
import shutil

CLEANUP_PRESETS = True  # Set to True to delete unused preset checkpoints

if CLEANUP_PRESETS and 'best_result' in locals() and best_result:
    print(f"\n{'='*70}")
    print("CLEANUP: Removing unused preset checkpoints")
    print(f"{'='*70}\n")
    
    winner_name = best_result['name']
    winner_ckpt_path = best_result['ckpt_path']
    print(f"🏆 Keeping only: {winner_name}\n")
    
    # Find all preset checkpoints (matching the naming pattern from cell 12)
    preset_checkpoints = list(CHECKPOINT_DIR.glob("debug_overfit_*_preset.pth"))
    
    deleted_count = 0
    freed_space = 0
    
    for ckpt_path in preset_checkpoints:
        # Check if this is the winner checkpoint
        if str(ckpt_path) == winner_ckpt_path:
            print(f"  ✅ Kept {ckpt_path.name} (winner)")
        else:
            # Delete the loser checkpoint
            file_size = ckpt_path.stat().st_size / (1024**3)  # Size in GB
            ckpt_path.unlink()
            freed_space += file_size
            deleted_count += 1
            print(f"  🗑️  Deleted {ckpt_path.name} ({file_size:.2f} GB)")
            
            # Also delete corresponding epochs folder
            epochs_dir = CHECKPOINT_DIR / f"{ckpt_path.stem}_epochs"
            if epochs_dir.exists():
                shutil.rmtree(epochs_dir)
                print(f"  🗑️  Deleted {epochs_dir.name}/ (epochs)")
    
    print(f"\n💾 Deleted {deleted_count} checkpoints, freed {freed_space:.2f} GB")
    print(f"{'='*70}\n")
elif CLEANUP_PRESETS:
    print("⚠️  No winner found yet. Run preset comparison first.")
else:
    print("ℹ️  Preset cleanup skipped (CLEANUP_PRESETS = False)")

In [ ]:
SKIP_OVERFIT = False
FORCE_RETRAIN = True  # Set True to retrain even if checkpoint exists

if not SKIP_OVERFIT:
    overfit_ckpt = CHECKPOINT_DIR / "debug_overfit_1song.pth"
    
    # Clean up old checkpoint if retraining
    if FORCE_RETRAIN and overfit_ckpt.exists():
        overfit_ckpt.unlink()
        # Also clean up epochs folder
        epochs_dir = CHECKPOINT_DIR / "debug_overfit_1song_epochs"
        if epochs_dir.exists():
            import shutil
            shutil.rmtree(epochs_dir)
        print("🗑️  Deleted old checkpoint and epochs")
    
    print(f"\n{'='*60}")
    print("Overfit Sanity Check")
    print(f"{'='*60}")
    checkpoint_exists = utils.check_checkpoint(overfit_ckpt, "Overfit Checkpoint")
    
    if not checkpoint_exists and len(mix_files_stage1) > 0:
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(DATA_DIR),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )
    elif len(mix_files_stage1) == 0:
        print("\n⚠️  No training data available.")
        print("   Set DOWNLOAD_DATA = True to download MUSDB18 dataset.")
        history_overfit = {}
    else:
        history_overfit = utils.run_overfit_1song(
            overfit_model=overfit_model,
            overfit_processor=overfit_processor,
            overfit_optimizer=overfit_optimizer,
            overfit_loss_fn=overfit_loss_fn,
            overfit_config=OVERFIT_CONFIG,
            cache_dir=str(DATA_DIR),
            save_path=str(overfit_ckpt),
            device=overfit_device,
        )
else:
    print("Skipping overfit test")
    history_overfit = {}

### View Overfit Results

In [ ]:
# Plot overfit learning curve from checkpoint or memory
overfit_ckpt_path = CHECKPOINT_DIR / "debug_overfit_1song.pth"

if overfit_ckpt_path.exists():
    utils.plot_loss_from_checkpoint(str(overfit_ckpt_path), title="Overfit Learning Curve (1 Song)")
elif history_overfit:
    utils.plot_loss_history(history_overfit, title="Overfit Learning Curve (1 Song)")
else:
    print("No overfit results available. Run the overfit training cell above.")

## Overfit Evaluation

Spectrograms and audio for the single overfitting song.


In [ ]:
# ============================================================================
# EVALUATE OVERFIT: VISUALIZE SEPARATION RESULTS
# ============================================================================

# Load overfit model checkpoint
overfit_ckpt_path = CHECKPOINT_DIR / "debug_overfit_1song.pth"

if overfit_ckpt_path.exists():
    print(f"\n{'='*70}")
    print("OVERFIT EVALUATION: Single Song Separation")
    print(f"{'='*70}\n")
    
    # Load checkpoint
    checkpoint = torch.load(overfit_ckpt_path, map_location=overfit_device)
    overfit_model.load_state_dict(checkpoint['model_state_dict'])
    overfit_model.eval()
    
    # Load the overfitting song (first training example)
    mix_dir = DATA_DIR / "stage1" / "train" / "mixture"
    tgt_dir = DATA_DIR / "stage1" / "train" / "target"
    
    if mix_dir.exists() and tgt_dir.exists():
        mix_files = sorted(mix_dir.glob("*.npy"))
        tgt_files = sorted(tgt_dir.glob("*.npy"))
        
        if len(mix_files) > 0:
            # Load first song
            mixture = np.load(mix_files[0])
            ground_truth = np.load(tgt_files[0])
            
            # Process with model
            mix_tensor = torch.FloatTensor(mixture).unsqueeze(0).unsqueeze(0).to(overfit_device)
            
            with torch.no_grad():
                pred_tensor = overfit_model(mix_tensor)
            
            prediction = pred_tensor.squeeze().cpu().numpy()
            
            # Plot spectrograms and play audio
            utils.plot_spectrograms_and_play_audio(
                mixture=mixture,
                prediction=prediction,
                ground_truth=ground_truth,
                sr=SAMPLE_RATE,
                title="Overfit Song: Mixture vs Prediction vs Ground Truth",
                show_audio=True
            )
            
            print("✅ Evaluation complete!")
        else:
            print("⚠️  No training files found")
    else:
        print("⚠️  Data directories not found. Run preprocessing first.")
else:
    print("⚠️  Overfit checkpoint not found. Run overfit training first.")

## Model Architecture

U-Net model structure and parameter count.

In [ ]:
# Model A architecture summary
model_summary = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=64,
    num_layers=4
).to(device)
print(model_summary)
del model_summary

## Setup Model

Use the winning architecture from preset comparison.

In [ ]:
# ============================================================================
# USE WINNING PRESET FOR FULL TRAINING
# ============================================================================

# Extract winner configuration from preset comparison
if 'best_result' in locals() and best_result:
    print(f"\n🏆 Using WINNER for full training: {best_result['name']}")
    print(f"   Final Val Loss: {best_result['final_val_loss']:.6f}")
    print(f"   Parameters: {best_result['params']:,}\n")
    
    # Find matching preset
    winner_preset = None
    for preset in PRESETS:
        if preset['name'] == best_result['name']:
            winner_preset = preset
            break
    
    if winner_preset:
        WINNER_CHUNK_DURATION = winner_preset['chunk_duration']
        WINNER_NUM_LAYERS = winner_preset['num_layers']
        WINNER_BASE_FILTERS = winner_preset['base_filters']
        
        # Update data directory to use winner's chunk size
        TRAINING_DATA_DIR = DATA_DIR / f"chunks_{WINNER_CHUNK_DURATION:.0f}s"
    else:
        print("⚠️  Could not find winner preset config, using defaults")
        WINNER_CHUNK_DURATION = 8.0
        WINNER_NUM_LAYERS = 4
        WINNER_BASE_FILTERS = 256
        TRAINING_DATA_DIR = DATA_DIR
else:
    print("⚠️  No preset winner found, using default config")
    WINNER_CHUNK_DURATION = 8.0
    WINNER_NUM_LAYERS = 4
    WINNER_BASE_FILTERS = 256
    TRAINING_DATA_DIR = DATA_DIR

# Build winning model for full training
TRAIN_CONFIG = utils.get_training_config()
processor = utils.AudioProcessor(device=device)

model = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=WINNER_BASE_FILTERS,
    num_layers=WINNER_NUM_LAYERS,
    batchnorm=True,
    dropout=0.1
).to(device)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=TRAIN_CONFIG['learning_rate'])

print(f"\n✅ Model setup complete:")
print(f"   Layers: {WINNER_NUM_LAYERS}")
print(f"   Base Filters: {WINNER_BASE_FILTERS}")
print(f"   Chunk Duration: {WINNER_CHUNK_DURATION}s")
print(f"   Data Directory: {TRAINING_DATA_DIR}")
print(f"   Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Train Model

Stage 1: Simple 2-source. Stage 2: Complex 4-source.

In [ ]:
SKIP_TRAINING = False  # Set to True to skip full training

if not SKIP_TRAINING:
    # Check if checkpoints exist
    ckpt_s1 = CHECKPOINT_DIR / f"full_stage1_{WINNER_CHUNK_DURATION:.0f}s.pth"
    ckpt_s2 = CHECKPOINT_DIR / f"full_stage2_{WINNER_CHUNK_DURATION:.0f}s.pth"
    
    print(f"\n{'='*60}")
    print(f"Full Training: {best_result['name'] if 'best_result' in locals() else 'Default Config'}")
    print(f"{'='*60}")
    
    s1_exists = utils.check_checkpoint(ckpt_s1, "Stage 1 Checkpoint")
    s2_exists = utils.check_checkpoint(ckpt_s2, "Stage 2 Checkpoint")
    
    # Check if winner's data exists
    stage1_dir = TRAINING_DATA_DIR / "stage1" / "train" / "mixture"
    if stage1_dir.exists():
        if not s1_exists or not s2_exists:
            print(f"\n🚀 Starting Full Training with {WINNER_CHUNK_DURATION:.0f}s chunks...")
            hist_s1, hist_s2 = utils.run_full_training(
                model=model,
                processor=processor,
                optimizer=optimizer,
                loss_fn=loss_fn,
                train_config=TRAIN_CONFIG,
                cache_dir=str(TRAINING_DATA_DIR),
                save_path_stage1=str(ckpt_s1),
                save_path_stage2=str(ckpt_s2),
                device=device,
            )
        else:
            print("\n✅ Both checkpoints exist. Training skipped.")
            hist_s1, hist_s2 = {}, {}
    else:
        print(f"\n⚠️  No training data found for {WINNER_CHUNK_DURATION:.0f}s chunks")
        print(f"   Expected location: {TRAINING_DATA_DIR}")
        print(f"   Run preprocessing for this chunk duration first.")
        hist_s1, hist_s2 = {}, {}
else:
    print("Skipping full training (SKIP_TRAINING = True)")
    hist_s1, hist_s2 = {}, {}

## View Training Results

Plot loss curves from saved checkpoint.

In [ ]:
# Plot training curves - reads from epoch folders if available!
import re
import matplotlib.pyplot as plt
from pathlib import Path

stage2_ckpt = CHECKPOINT_DIR / "full_stage2.pth"
stage2_epochs_dir = CHECKPOINT_DIR / "full_stage2_epochs"

# Try epoch folder first (has complete history)
if stage2_epochs_dir.exists():
    print(f"📁 Reading from: {stage2_epochs_dir.name}")
    
    epoch_files = sorted(stage2_epochs_dir.glob("epoch_*.txt"))
    train_losses = []
    val_losses = []
    
    for epoch_file in epoch_files:
        with open(epoch_file, 'r') as f:
            content = f.read()
            train_match = re.search(r'Train Loss = ([\d.]+)', content)
            val_match = re.search(r'Val Loss = ([\d.]+)', content)
            if train_match and val_match:
                train_losses.append(float(train_match.group(1)))
                val_losses.append(float(val_match.group(1)))
    
    if train_losses:
        print(f"✅ Found {len(train_losses)} epochs\n")
        
        fig, ax = plt.subplots(figsize=(12, 6))
        epochs = range(1, len(train_losses) + 1)
        ax.plot(epochs, train_losses, 'o-', label='Train Loss', linewidth=2, markersize=6)
        ax.plot(epochs, val_losses, 's--', label='Val Loss', linewidth=2, markersize=6)
        ax.set_title("Full Training: Stage 2", fontsize=14, fontweight='bold')
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel("Loss", fontsize=12)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        print(f"📊 Stats: Epochs={len(train_losses)} | Best Val={min(val_losses):.6f} (epoch {val_losses.index(min(val_losses))+1})")
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ Epoch files found but couldn't parse them")
        
elif stage2_ckpt.exists():
    utils.plot_loss_from_checkpoint(str(stage2_ckpt), title="Full Training: Stage 2")
elif hist_s2:
    utils.plot_loss_history(hist_s2, title="Full Training: Stage 2")
else:
    print("No training results available. Run the full training cell above.")

## Full Training Evaluation

Interactive menu to select test songs and evaluate separated vocals with spectrograms and audio.

In [ ]:
# ============================================================================
# FULL TRAINING EVALUATION: INTERACTIVE SONG SELECTOR
# ============================================================================

stage2_ckpt = CHECKPOINT_DIR / f"full_stage2_{WINNER_CHUNK_DURATION:.0f}s.pth"

if stage2_ckpt.exists():
    print(f"\n{'='*70}")
    print("FULL TRAINING EVALUATION: Interactive Song Selector")
    print(f"{'='*70}\n")
    
    # Load checkpoint into model
    checkpoint = torch.load(stage2_ckpt, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"✅ Loaded checkpoint: {stage2_ckpt.name}\n")
    
    # Create interactive evaluator
    utils.evaluate_with_song_selector(
        model=model,
        processor=processor,
        data_dir=TRAINING_DATA_DIR,
        sr=SAMPLE_RATE,
        device=device
    )
else:
    print(f"⚠️  Full training checkpoint not found: {stage2_ckpt}")
    print("   Run the full training cell above first.")

## Cleanup: Delete Unused Chunk Data

Free up disk space by removing chunk sizes not used by the winning model.

In [ ]:
# ============================================================================
# CLEANUP: DELETE UNUSED CHUNK DATA
# ============================================================================
import shutil

DELETE_UNUSED_CHUNKS = False  # Set to True to free up disk space

if DELETE_UNUSED_CHUNKS and 'WINNER_CHUNK_DURATION' in locals():
    print(f"\n{'='*70}")
    print("CLEANUP: Deleting unused chunk data")
    print(f"{'='*70}\n")
    
    # All possible chunk durations
    all_chunk_durations = [6.0, 8.0, 10.0, 12.0]
    
    # Remove winner from list
    unused_durations = [d for d in all_chunk_durations if d != WINNER_CHUNK_DURATION]
    
    print(f"🏆 Keeping: {WINNER_CHUNK_DURATION:.0f}s chunks (winner)")
    print(f"🗑️  Deleting: {', '.join([f'{d:.0f}s' for d in unused_durations])} chunks\n")
    
    total_freed = 0
    for duration in unused_durations:
        chunk_dir = DATA_DIR / f"chunks_{duration:.0f}s"
        
        if chunk_dir.exists():
            # Calculate size before deletion
            dir_size = sum(f.stat().st_size for f in chunk_dir.rglob('*') if f.is_file())
            size_gb = dir_size / (1024**3)
            
            # Delete directory
            shutil.rmtree(chunk_dir)
            total_freed += size_gb
            
            print(f"  ✅ Deleted chunks_{duration:.0f}s/ ({size_gb:.2f} GB)")
        else:
            print(f"  ⏭️  chunks_{duration:.0f}s/ not found (skipped)")
    
    print(f"\n💾 Total disk space freed: {total_freed:.2f} GB")
    print(f"{'='*70}\n")
else:
    if DELETE_UNUSED_CHUNKS:
        print("⚠️  No winner found. Run preset comparison first.")
    else:
        print("ℹ️  Cleanup skipped (DELETE_UNUSED_CHUNKS = False)")
        print("   Set to True to free up disk space after training completes.")